In [15]:
!pip install bertopic sentence-transformers umap-learn hdbscan

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 95.3 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 93.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 618.0/618.0 kB 59.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 84.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 86.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 93.0 MB/s  0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 84.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 90.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.6/801.6 kB 64.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 802.1/802.1 kB 67.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 94.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━

In [ ]:
# Si hdbscan marche pas on peut utiliser ça

from sklearn.cluster import HDBSCAN
from bertopic.cluster import BaseCluster
"""
class SklearnHDBSCAN(BaseCluster):
    def __init__(self, *args, **kwargs):
        self.model = HDBSCAN(*args, **kwargs)
    
    def fit(self, X):
        self.model.fit(X)
        self.labels_ = self.model.labels_
        return self
    
    def fit_predict(self, X):
        self.labels_ = self.model.fit_predict(X)
        return self.labels_
hdbscan_model = SklearnHDBSCAN(
    min_cluster_size=50,      # bigger = fewer, larger topics
    min_samples=10,            # higher = more conservative clustering
    cluster_selection_epsilon=0.1,  # merge nearby clusters
    cluster_selection_method='leaf' # fewer, more specific clusters
)
"""

Récupération des fichiers CSV

In [ ]:
import pandas as pd
import os
import s3fs



fs = s3fs.S3FileSystem(
    client_kwargs={'endpoint_url': 'https://'+'minio.lab.sspcloud.fr'},
    key = os.environ["AWS_ACCESS_KEY_ID"], 
    secret = os.environ["AWS_SECRET_ACCESS_KEY"], 
    token = os.environ["AWS_SESSION_TOKEN"])

s3_folder = "s3://lab/PESSD/csv/patin_art"

files = fs.glob(f"{s3_folder}/*.csv")

for file in files:
    fs.get(file, "data/"+(file.removeprefix("lab/PESSD/csv/patin_art/")))

In [11]:
df = pd.DataFrame()
for file in os.listdir("data") : 
    temp = pd.read_csv("data/"+file)
    df = pd.concat([df,temp])


Nettoyage de la data

In [12]:
import re

# Removing music and noise (only technical commentary)
df = df[(df["main_g"]=="female")|(df["main_g"]=="male")]

# text to sentences
df["text"] = df["text"].apply(lambda x : x.split(". "))
df = df.explode("text")

# Removes all words with capitals
# A éviter car détruit du sens

#df["text"] = df["text"].apply(lambda x : re.sub(r"\s*[A-Z]\w*\s*", " ", x).strip())


On ajoute les métadonnées

In [13]:
df["ID"] = df["audio_file"].str.replace("_wav.wav","")
df = df.merge(pd.read_csv("metadata.csv"), on="ID")

In [14]:
df

,start,stop,text,main_speaker,main_g,audio_file,ID,g_ath
0,7.08,17.88,les ...,SPEAKER_02,female,ELC_wav.wav,ELC,M
1,7.08,17.88,Les mamansament font lui,SPEAKER_02,female,ELC_wav.wav,ELC,M
2,18.04,4.98,des,SPEAKER_00,female,ELC_wav.wav,ELC,M
3,19.16,52.68,iniciants,SPEAKER_02,female,ELC_wav.wav,ELC,M
4,19.16,52.68,Ils n'entraînent pas et n'en peuvent pas,SPEAKER_02,female,ELC_wav.wav,ELC,M
...,...,...,...,...,...,...,...,...
7855,1916.16,1949.48,Et passer le cut,SPEAKER_04,female,DR2_wav.wav,DR2,F
7856,1916.16,1949.48,"Merci Nathalie et Péchala, merci Annick Dumont",SPEAKER_04,female,DR2_wav.wav,DR2,F
7857,1916.16,1949.48,"Merci Marie-Christine, Marie",SPEAKER_04,female,DR2_wav.wav,DR2,F
7858,1916.16,1949.48,On se retrouve demain pour de nouvelles aventures,SPEAKER_04,female,DR2_wav.wav,DR2,F


In [19]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
import hdbscan

# Stopwords list
STOPWORDS = [x.strip() for x in open('stopwords.txt').readlines()]

# Docs 
docs = df["text"]

# Vectorizer for topic representation
vectorizer_model = CountVectorizer(
    stop_words=STOPWORDS,
    ngram_range=(1,2),
    min_df=2,
    token_pattern=r"\b(?!\w*[A-Z])\w+\b" # theoreticaly removes words with capital letters once embeddings are done
)

# To control for number and size of clusters
hdbscan_model = hdbscan.HDBSCAN(
    min_cluster_size=50,      # bigger = fewer, larger topics
    min_samples=10,            # higher = more conservative clustering
    cluster_selection_epsilon=0.1,  # merge nearby clusters
    cluster_selection_method='leaf', # fewer, more specific clusters
    prediction_data=True # important !!
)

# Load CamemBERT embedding model
embedding_model = SentenceTransformer("dangvantuan/sentence-camembert-base", device="cuda") # remove device if CPU

# Influence choice of themes
seed =  [
["émotion","larme","ému","joie","tristesse"],
["famille","parents","ami", "personnel", "conjoint"]]

# Build BERTopic model
topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    hdbscan_model=hdbscan_model,
    language="french",
    calculate_probabilities=True,
    verbose=True,
    seed_topic_list=seed    
)

# Fit the model
topics, probs = topic_model.fit_transform(docs)

# Show discovered topics
print(topic_model.get_topic_info())

# Visualize topics
#topic_model.visualize_topics()

# Visualize topic hierarchy
topic_model.visualize_hierarchy()

#keybertopic, extrait phrases saillantes
#tester lda
#virer phrases nulles et refaire tourner
#pas virer -1 sauf si faible échantillon et checker
#checker si semi-superviser pour trouver thèmes

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2882.61it/s]
CamembertModel LOAD REPORT from: dangvantuan/sentence-camembert-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-03-22 15:42:53,942 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 246/246 [00:12<00:00, 20.28it/s]
2026-03-22 15:43:06,156 - BERTopic - Embedding - Completed ✓
2026-03-22 15:43:06,157 - BERTopic - Guided - Find embeddings highly related to seeded topics.
Batches: 100%|██████████| 1/1 [00:00<00:00, 100.88it/s]
2026-03-22 15:43:06,233 - BERTopic - Guided - Completed ✓
2026-03-22 15:43:06,318 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-22 15:43:45,850 - BERTopic - Dimensionality - Completed ✓
2026-03-22 15:43:45,852 - BER

    Topic  Count                                             Name  \
0      -1   3188          -1_programme_vraiment_points_évidemment   
1       0    307               0_musique_chorégraphie_danse_choix   
2       1    305            1_incroyable_parfait_magnifique_super   
3       2    282                 2_couple_couples_canadiens_japon   
4       3    281          3_patinage_patineur_patineurs_patineuse   
5       4    254                          4_aïe_aïe aïe_joie_émue   
6       5    204                      5_saut_sauts_lutz_quadruple   
7       6    178       6_provisoire_place_place provisoire_podium   
8       7    163                      7_sûre_regardons_voir_verra   
9       8    152            8_niveau_technique_4_niveau technique   
10      9    149           9_europe_monde_champion_champion monde   
11     10    127             10_glace_danse glace_danse_patinoire   
12     11    122                 11_triple_petit_saut_combinaison   
13     12    120              12_p

On regroupe les phrases par topic

In [21]:
tp = pd.DataFrame({
    "text": docs,
    "topic": topics,
    "topic_prob": probs.max(axis=1)  # max probability per text
})
tp["topic"].value_counts()

topic
-1     3188
 0      307
 1      305
 2      282
 3      281
 4      254
 5      204
 6      178
 7      163
 8      152
 9      149
 10     127
 11     122
 12     120
 13     118
 15     105
 14     105
 16     103
 17     100
 18      92
 19      88
 20      87
 21      87
 22      86
 23      82
 25      78
 24      78
 26      72
 27      68
 28      64
 29      63
 30      60
 31      58
 32      57
 33      56
 35      55
 34      55
 36      54
 37      54
 38      53
 39      50
Name: count, dtype: int64

In [22]:
df = df.merge(tp, on="text")
df

,start,stop,text,main_speaker,main_g,audio_file,ID,g_ath,topic,topic_prob
0,7.08,17.88,les ...,SPEAKER_02,female,ELC_wav.wav,ELC,M,-1,0.052528
1,7.08,17.88,Les mamansament font lui,SPEAKER_02,female,ELC_wav.wav,ELC,M,2,1.000000
2,18.04,4.98,des,SPEAKER_00,female,ELC_wav.wav,ELC,M,-1,0.048393
3,19.16,52.68,iniciants,SPEAKER_02,female,ELC_wav.wav,ELC,M,12,1.000000
4,19.16,52.68,Ils n'entraînent pas et n'en peuvent pas,SPEAKER_02,female,ELC_wav.wav,ELC,M,26,1.000000
...,...,...,...,...,...,...,...,...,...,...
15025,1916.16,1949.48,Et passer le cut,SPEAKER_04,female,DR2_wav.wav,DR2,F,-1,0.049630
15026,1916.16,1949.48,"Merci Nathalie et Péchala, merci Annick Dumont",SPEAKER_04,female,DR2_wav.wav,DR2,F,17,1.000000
15027,1916.16,1949.48,"Merci Marie-Christine, Marie",SPEAKER_04,female,DR2_wav.wav,DR2,F,2,0.144693
15028,1916.16,1949.48,On se retrouve demain pour de nouvelles aventures,SPEAKER_04,female,DR2_wav.wav,DR2,F,-1,0.022534


In [23]:
# Sentences of topic i

for k in df[df["topic"]==1]["text"] : 
    print(k)

C'est ça qui est hyper impressionnant
Tout était incroyable et fantastique.
Jamais de la vie
Il est super
La sensation, elle doit être incroyable
Et brillant
C'était un truc pour...
Triple bout, parfait.
Et c'est très bien
C'est très très rare ce que l'on vient de voir
Ça aussi, c'est incroyable
C'est très rare
Mais c'est quand même exceptionnel.
Ils arrivent vraiment à nous embarquer dans leur histoire.
J'adore cette image arrêtée, c'est beau
Je l'adore, c'est super.
Le score, incroyable
Même là, c'est limite.
Très souvent, tout cas Daniel.
Parfois ça ne passe pas, mais quand ça passe, qu'est-ce que c'est bon
C'est extra
Ça c'est super
No, I wish the sea wonder
C'est extraordinaire
C'est extraordinaire
Moi, j'ai adoré
C'était une pure merveille
Tout est subtil
C'est absolument divin
J'ai adoré
Donc ça serait extraordinaire
Sur le reste c'est magnifique
Mais elle va être excellente
Alors moi je l'adore à elle
portée là, qui est magnifique
Niveau 4 oui, mais surtout c'est original, c'es

Tests de stabilité

In [ ]:
# Main words of current model themes

reference = set()
for topic_id in topic_model.get_topics():
    if topic_id == -1:
        continue
    words = [w for w, _ in topic_model.get_topic(topic_id)[:10]]
    reference.add(tuple(sorted(words)))
reference

{('1',
  '8',
  'attention',
  'bonsoir',
  'fantastique',
  'ok',
  'oui',
  'oui oui',
  'oui thème',
  'thème'),
 ('100',
  '2ème',
  '2ème note',
  'monte',
  'monter',
  'note',
  'notes',
  'score',
  'scoret',
  'scorez'),
 ('102', '102 55', '1m75', '46', '55', '62', '64', '70', '92', 'mesure'),
 ('2',
  '23',
  'attention',
  'bascule',
  'famille',
  'imagine',
  'maxime',
  'oui',
  'parfait',
  'suspense'),
 ('2024',
  'champion',
  'champion monde',
  'championnat',
  'championnat europe',
  'championnats',
  'championnats europe',
  'europe',
  'monde',
  'titre'),
 ('2e',
  'classement',
  'instant',
  'place',
  'place provisoire',
  'podium',
  'podium provisoire',
  'prennent',
  'provisoire',
  'tête'),
 ('3',
  '3 4',
  '4',
  '4 tours',
  'demi',
  'tour',
  'tour tours',
  'tours',
  'tours demi',
  'tours tours'),
 ('4',
  'maximum',
  'niveau',
  'niveau 4',
  'niveau maximum',
  'niveau technique',
  'niveaux',
  'technique',
  'technique niveau',
  'techniques'

On run 10 fois le même modèle à 80% du corpus pour voir la stabilité des thèmes

In [ ]:
from sklearn.utils import resample
from collections import Counter

# List of words for each topic for each model
all_topics_words = []

# Number of runs
nb_iter = 10

for i in range(nb_iter):

    # First, resample 80% of docs
    sample = resample(docs, n_samples=int(len(docs) * 0.8), random_state=i)

    # Same model specification
    model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    hdbscan_model=hdbscan_model,
    language="french",
    calculate_probabilities=True,
    verbose=True,
    seed_topic_list=seed    
)
    model.fit(sample)
    
    # Top 10 words per topic
    top_words = set()
    for topic_id in model.get_topics():
        if topic_id == -1:
            continue
        words = [w for w, _ in model.get_topic(topic_id)[:10]]
        top_words.add(tuple(sorted(words)))
    
    all_topics_words.append(top_words)


# Jaccard similarity index
def jaccard(t1, t2):
    s1, s2 = set(t1), set(t2)
    return len(s1 & s2) / len(s1 | s2)

# Topic stability
"""
Measures similarity between reference topic and topics found in other runs.

Thresholds for stability are determined by Jaccard index (threshold) and number of occurences (min_run)
"""
def is_stable(topic, all_runs, threshold=0.2, min_runs=7):
    count = 0
    for run_topics in all_runs:
        if any(jaccard(topic, t) >= threshold for t in run_topics):
            count += 1
    return count >= min_runs


stable_topics = [t for t in reference if is_stable(t, all_topics_words)]

print(f"Thèmes stables (≥7 runs sur 10) : {len(stable_topics)}")
for t in stable_topics:
    print(t)


In [19]:
from sklearn.utils import resample
from collections import Counter
from sklearn.cluster import HDBSCAN
from bertopic import BERTopic
from bertopic.cluster import BaseCluster
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
# Modèle de référence sur le corpus complet
ref_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=CountVectorizer(stop_words=STOPWORDS, ngram_range=(1,2), min_df=2, token_pattern=r"\b(?!\w*[A-Z])\w+\b"),
    hdbscan_model=SklearnHDBSCAN(min_cluster_size=50, min_samples=10, cluster_selection_epsilon=0.1, cluster_selection_method='leaf'),
    language="french",
    calculate_probabilities=True,
    verbose=True,
    seed_topic_list=seed
)
ref_model.fit(df["text"])

reference = set()
for topic_id in ref_model.get_topics():
    if topic_id == -1:
        continue
    words = [w for w, _ in ref_model.get_topic(topic_id)[:10]]  # 10 mots
    reference.add(tuple(sorted(words)))

# Bootstrap sur sous-échantillons
all_topics_words = []
for i in range(10):
    sample = resample(df["text"], n_samples=int(len(df["text"]) * 0.8), random_state=i)
    
    model_i = BERTopic(
        embedding_model=embedding_model,
        vectorizer_model=CountVectorizer(stop_words=STOPWORDS, ngram_range=(1,2), min_df=2, token_pattern=r"\b(?!\w*[A-Z])\w+\b"),
        hdbscan_model=SklearnHDBSCAN(min_cluster_size=50, min_samples=10, cluster_selection_epsilon=0.1, cluster_selection_method='leaf'),
        language="french",
        calculate_probabilities=True,
        verbose=True,
    )
    model_i.fit(sample)
    
    top_words = set()
    for topic_id in model_i.get_topics():
        if topic_id == -1:
            continue
        words = [w for w, _ in model_i.get_topic(topic_id)[:10]]  # 10 mots
        top_words.add(tuple(sorted(words)))
    
    all_topics_words.append(top_words)

# Faire un Jaccard entre chaque run et compter les thèmes stables
def jaccard(t1, t2):
    s1, s2 = set(t1), set(t2)
    return len(s1 & s2) / len(s1 | s2)

def is_stable(topic, all_runs, threshold=0.3, min_runs=7):
    count = 0
    for run_topics in all_runs:
        if any(jaccard(topic, t) >= threshold for t in run_topics):
            count += 1
    return count >= min_runs
# Stabilité : thèmes du modèle référence présents dans ≥7 runs bootstrap
stable_topics = [t for t in reference if is_stable(t, all_topics_words, threshold=0.2, min_runs=7)]
print(f"Thèmes stables (≥7 runs sur 10) : {len(stable_topics)}")
for t in stable_topics:
    print(t)

2026-03-21 18:29:08,986 - BERTopic - Embedding - Transforming documents to embeddings.
Batches:   0%|          | 0/246 [00:00<?, ?it/s]

Batches: 100%|██████████| 246/246 [00:13<00:00, 18.09it/s]
2026-03-21 18:29:22,682 - BERTopic - Embedding - Completed ✓
2026-03-21 18:29:22,683 - BERTopic - Guided - Find embeddings highly related to seeded topics.
Batches: 100%|██████████| 1/1 [00:00<00:00, 102.34it/s]
2026-03-21 18:29:22,835 - BERTopic - Guided - Completed ✓
2026-03-21 18:29:22,846 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-21 18:30:06,666 - BERTopic - Dimensionality - Completed ✓
2026-03-21 18:30:06,671 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-21 18:30:07,171 - BERTopic - Cluster - Completed ✓
2026-03-21 18:30:07,181 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-21 18:30:07,427 - BERTopic - Representation - Completed ✓
2026-03-21 18:30:07,707 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 197/197 [00:10<00:00, 18.02it/s]
2026-03-21 18:30:18,706 - BERTopic - Embeddi

Thèmes stables (≥7 runs sur 10) : 20
('canada', 'canada france', 'chine pologne', 'france', 'france tête', 'instant', 'italie', 'place', 'rappelle france', 'tête')
('2ème', '2ème note', 'monte', 'monter', 'note', 'notes', 'score', 'scoret', 'scorez', 'énormément')
('60', '71', 'battent', 'battu', 'meilleur', 'meilleur score', 'record', 'saison', 'score', 'score saison')
('classement', 'instant', 'place', 'place provisoire', 'podium', 'podium provisoire', 'prennent', 'provisoire', 'sato', 'tête')
('4', 'annick niveau', 'maximum', 'niveau', 'niveau 4', 'niveau maximum', 'niveau technique', 'niveaux', 'technique', 'éléments')
('90', 'années', 'chorégraphe', 'chorégraphie', 'chorégraphies', 'chorégraphique', 'dance', 'danse', 'rythme', 'rythme danse')
('5', '5 points', 'bonus', 'bonus malus', 'donner', 'juges', 'jury', 'malus', 'points', 'élément')
('championne', 'championne monde', 'japon', 'kajiyama', 'kaori', 'kaori sakamoto', 'monde', 'sakamoto', 'yuma', 'yuma kajiyama')
('combinaison'

In [9]:
# Install dependencies if needed
# pip install bertopic sentence-transformers umap-learn hdbscan

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
from sklearn.cluster import HDBSCAN
from bertopic.cluster import BaseCluster

class SklearnHDBSCAN(BaseCluster):
    def __init__(self, *args, **kwargs):
        self.model = HDBSCAN(*args, **kwargs)
    
    def fit(self, X):
        self.model.fit(X)
        self.labels_ = self.model.labels_
        return self
    
    def fit_predict(self, X):
        self.labels_ = self.model.fit_predict(X)
        return self.labels_
hdbscan_model = SklearnHDBSCAN(
    min_cluster_size=50,      # bigger = fewer, larger topics
    min_samples=10,            # higher = more conservative clustering
    cluster_selection_epsilon=0.1,  # merge nearby clusters
    cluster_selection_method='leaf' # fewer, more specific clusters
)

# Load CamemBERT embedding model
embedding_model = SentenceTransformer("dangvantuan/sentence-camembert-base", device="cuda")

STOPWORDS = [x.strip() for x in open('stopwords.txt').readlines()]

# Vectorizer for topic representation
vectorizer_model = CountVectorizer(
    stop_words=STOPWORDS,
    ngram_range=(1,2),
    min_df=2
)
#umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, random_state=42)
seed =  [
["émotion","larme","ému","joie","tristesse"],
["famille","parents","ami", "personnel", "conjoint"]]
""",
["attaque","montée","effort","énergie","relance","sprint"],        # tactique
["médaille","podium","victoire","champion","record","victoire","classement","premier","deuxième","troisième"],              # résultats
["bravo","chapeau","incroyable","monstrueux","magnifique","énorme"],        # admiration
["tir","cible","pénalité","debout","couché","faute"],                      # technique
["âge","ans","carrière","champion du monde","olympique","saison"],
["rapide","forme","expérience","solide","rapide","bon"],          # profil athlète
["devant","revient","prend la tête","écart","seconde", "temps","tour","intermédiaire","course","piste"] 
]"""
# Build BERTopic model
topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    hdbscan_model=hdbscan_model,
    language="french",
    calculate_probabilities=True,
    verbose=True,
    seed_topic_list=seed    
)

# Fit the model
topics, probs = topic_model.fit_transform(df["text"])

# Show discovered topics
print(topic_model.get_topic_info())

# Show keywords for a specific topic
print(topic_model.get_topic(0))

# Visualize topics
topic_model.visualize_topics()

# Visualize topic hierarchy
topic_model.visualize_hierarchy()

#keybertopic, extrait phrases saillantes
#tester lda
#virer phrases nulles et refaire tourner
#pas virer -1 sauf si faible échantillon et checker
#checker si semi-superviser pour trouver thèmes

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3988.47it/s]
CamembertModel LOAD REPORT from: dangvantuan/sentence-camembert-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-03-21 18:17:45,341 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 246/246 [00:12<00:00, 19.01it/s]
2026-03-21 18:17:58,369 - BERTopic - Embedding - Completed ✓
2026-03-21 18:17:58,370 - BERTopic - Guided - Find embeddings highly related to seeded topics.
Batches: 100%|██████████| 1/1 [00:00<00:00, 16.60it/s]
2026-03-21 18:17:58,642 - BERTopic - Guided - Completed ✓
2026-03-21 18:17:58,643 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-21 18:18:42,881 - BERTopic - Dimensionality - Completed ✓
2026-03-21 18:18:42,882 - BERT

    Topic  Count                                               Name  \
0      -1   3249             -1_programme_vraiment_points_technique   
1       0    283                       0_axel_bras_position_spirale   
2       1    282                  1_couple_couples_canadiens_canada   
3       2    280            2_patinage_patineur_patineurs_patineuse   
4       3    260       3_incroyable_parfait_magnifique_exceptionnel   
5       4    256                         4_aïe_aïe aïe_joie_sourire   
6       5    209                 5_chorégraphie_musique_danse_choix   
7       6    181                        6_vitesse_triple_petit_saut   
8       7    175                        7_voir_sûre_regardons_verra   
9       8    169         8_provisoire_place_place provisoire_podium   
10      9    136                 9_oui oui_oui_sûr_oui complètement   
11     10    133               10_glace_danse glace_danse_patinoire   
12     11    132                 11_limite_parfait_attention_maxime   
13    

In [11]:
t = pd.DataFrame({
    "text": df['text'],
    "topic": topics
    #"topic_prob": probs.max(axis=1)  # max probability per text
})

In [ ]:
t = pd.DataFrame({
    "text": df['text'],
    "topic": topics
    #"topic_prob": probs.max(axis=1)  # max probability per text
})
t["topic"].value_counts()

topic
-1     3249
 0      283
 1      282
 2      280
 3      260
 4      256
 5      209
 6      181
 7      175
 8      169
 9      136
 10     133
 11     132
 12     125
 13     122
 14     118
 15     105
 16     104
 17     103
 18     100
 19      98
 20      93
 21      92
 22      91
 23      78
 24      75
 25      75
 26      66
 27      61
 29      59
 28      59
 30      58
 31      57
 32      56
 33      55
 34      54
 36      54
 35      54
 37      53
 38      50
Name: count, dtype: int64

In [67]:
topic_model.get_topic(4,full=True)

False

In [ ]:
#1 : 4, 17, 18 retirés
#2 : 

t = pd.DataFrame({
    "text": df['text'],
    "topic": topics
    #"topic_prob": probs.max(axis=1)  # max probability per text
})
t["topic"].value_counts()
for k in t[t["topic"]==4]["text"] : 
    print(k)

On est très fiers
Vu leur joie, effectivement
Oui, soyez contents.
Il fait des investissements dans l'immobilier
Ils sont contents
Et d'ailleurs, ça me dépensait à quelque chose
Regardez le visage du partenaire, il souffre
Il doit mourir.
Annick, vous avez été émue par ce programme ?
Oui, j'ai été émue incontestablement
Elle décompense, elle dit.
Tout paraît tellement simple.
Annick, vous en avez pensé quoi ? Mais c'est compliqué pour la France.
Elle ne l'est pas du tout, du tout
Il a une patate, ça se transmet, on sent qu'il aime ça
Mais ce n'est pas ce qu'on nous souhaite, le pot
Et le petit sourire
Kajiyama qui avait remporté la médaille de bronze au dernier championnat du monde et était extrêmement extrêmement déçu
Il disait voilà j'ai loupé parce que j'ai plus confiance en moi et finalement vous voyez c'est pas le cas
Il a ça dans le sang
Il a raté sa vie, c'est tout
Très fâchée, c'est un grand mot.
Et effectivement j'étais surprise, mais pas complètement, puisque ce n'est pas la 

In [28]:
mask = t[(t["topic"]!=4)&(t["topic"]!=17)&(t["topic"]!=18)]["text"]

In [29]:
d2 = d.merge(mask, on="text")

In [64]:
d2.merge(t,on="text")[d2.merge(t,on="text")["topic"]==18]

,Unnamed: 0,start,stop,text,main_speaker,main_g,audio_file,topic,topic_prob
498,398,4755.72,4757.72,C'est une belle histoire pour lui.,SPEAKER_05,female,BI_wav.wav,18,1.00000
588,515,6473.40,6476.36,"Il y a des leçons à tirer, malgré la déception...",SPEAKER_11,male,BI_wav.wav,18,1.00000
605,11,424.80,440.80,Juste le temps peut-être de saluer dans la raq...,SPEAKER_11,male,BIF_wav.wav,18,0.14816
937,342,4304.68,4312.68,C'est les belles histoires des jeux. Les belle...,SPEAKER_06,female,BIF_wav.wav,18,1.00000
948,353,4474.68,4496.68,"Oui, c'est vraiment... Bien évidemment, Trista...",SPEAKER_11,male,BIF_wav.wav,18,1.00000
1005,56,814.00,820.00,C'est long une Olympiade Avec toutes les émoti...,SPEAKER_04,male,BMH_wav.wav,18,1.00000
1041,97,1299.90,1307.90,La santé c'est le plus important. Elle peut êt...,SPEAKER_03,male,BMH_wav.wav,18,1.00000
1048,104,1369.90,1373.90,"Quand on connaît ses qualités, ça fend le cœur...",SPEAKER_04,female,BMH_wav.wav,18,1.00000
1621,173,1963.72,1972.56,"Va se laisser prendre par les émotions. Allez,...",SPEAKER_01,male,BMF_wav.wav,18,1.00000
1954,246,2668.36,2672.36,Oh les belles émotions.,SPEAKER_04,music,BMF_wav.wav,18,1.00000


In [48]:
topic_model.get_topic(-1)

[('tir', np.float64(0.019705384158766548)),
 ('course', np.float64(0.01810100205230608)),
 ('julia', np.float64(0.016166922702569482)),
 ('secondes', np.float64(0.01581559885555621)),
 ('petit', np.float64(0.015216028128888991)),
 ('vraiment', np.float64(0.014750896397992897)),
 ('faut', np.float64(0.014264980364740047)),
 ('allez', np.float64(0.014103260774615873)),
 ('20', np.float64(0.01389915649281138)),
 ('aller', np.float64(0.013604787866193726))]

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer("dangvantuan/sentence-camembert-base")


sentence_embeddings = model.encode(d2["text"])
theme_embeddings = model.encode(themes)

sim = cosine_similarity(sentence_embeddings, theme_embeddings)
labels = sim.argmax(axis=1)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2362.30it/s]
CamembertModel LOAD REPORT from: dangvantuan/sentence-camembert-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [136]:
d2["labels"]=labels

In [147]:
d2[d2["labels"]==8]["text"]

0       En endroit, on est parti pour vivre encore un ...
3       Moi je vais vous donner les numéros de Dossard...
4       Ouais l'équivalent à peu près de deux tours et...
7       Il a un rôle important dans cette équipe. Donc...
9       On discutait hier justement dans la reconnaiss...
                              ...                        
1427    Et on prend le temps de savourer. Et nous auss...
1428    Elle vient de passer derrière le pas de tir. E...
1434    Simon l'a dit, Simon Fourcade, l'entraîneur de...
1440    La Suède, qui revient à 3 secondes dès Nord-Vé...
1441    Ça va pas le mettre en confiance pour le début...
Name: text, Length: 444, dtype: object

In [131]:
themes = []
for i in seed : 
    themes.append(" ".join(i))
themes

['émotion larme ému joie triste histoire hommage',
 'famille parents ami personnelle',
 'attaque montée effort énergie relance sprint',
 'médaille podium victoire champion record victoire classement premier deuxième troisième',
 'bravo chapeau incroyable monstrueux magnifique énorme',
 'tir cible pénalité debout couché faute',
 'âge ans carrière champion du monde olympique saison',
 'rapide forme expérience solide rapide bon',
 'devant revient prend la tête écart seconde temps tour intermédiaire course piste']

In [65]:
test = topic.merge(d, on="text")

In [67]:
test[(test["topic"]==10)|(test["topic"]==15)]

,text,topic,topic_prob,start,stop,main_speaker,main_g,audio_file
8,Pour vous en dire un petit peu plus sur la pis...,10,1.000000,289.60,345.80,SPEAKER_05,female,BI_wav.wav
9,On discutait hier justement dans la reconnaiss...,10,0.124827,345.80,356.40,SPEAKER_12,male,BI_wav.wav
11,Après c'est vrai que cette année il y a des st...,15,1.000000,374.20,386.60,SPEAKER_05,female,BI_wav.wav
13,"C'est un athlète solide, le jeune Suisse qui p...",15,0.238053,421.20,434.20,SPEAKER_12,male,BI_wav.wav
25,C'est un athlète qui a énormément d'expérience...,15,0.219121,604.20,622.20,SPEAKER_12,male,BI_wav.wav
26,"Cette année, il est un tout petit peu moins bo...",15,1.000000,622.20,634.20,SPEAKER_05,female,BI_wav.wav
28,"Fabien, c'est vraiment un skieur qui fait part...",15,1.000000,677.20,709.20,SPEAKER_05,female,BI_wav.wav
78,"Là, ça fait mal. On voit dans le regard qu'il ...",10,0.364798,1586.20,1595.20,SPEAKER_12,female,BI_wav.wav
93,On a l'impression qu'il y a les jambes qui tre...,10,1.000000,1724.20,1734.20,SPEAKER_05,female,BI_wav.wav
639,"Comme on l'avait dit pour le relais mixte, c'e...",10,0.152644,4025.84,4036.72,SPEAKER_05,female,BI_wav.wav
